# CerberusAI v2 - Merge, GGUF y evaluacion en Google Colab

Continua una corrida de `train_cerberus_colab.ipynb` sin usar el PC:

1. Merge del adapter LoRA sobre `Qwen2.5-Coder-7B-Instruct` (fp16, CPU con mucha RAM).
2. Conversion directa a **GGUF Q8_0** con llama.cpp (sin compilarlo).
3. Ollama dentro de Colab: `cerberusai-v2`, base `qwen2.5-coder:7b-instruct` y `cerberusai-v1` (opcional).
4. Oraculo Java (`audit.CompiladorCLI`) compilado con `javac` + `TestSuite`.
5. `run_eval.py`: tasa de compilacion y exact-match sobre las tareas `generation` de `eval.jsonl`.

Todo queda en `MyDrive/CerberusAI/runs/<RUN_ID>/`: `gguf/` (GGUF + `gguf_info.json`) y `eval/`.
**Runtime: A100** (necesita >= 40 GB de RAM para el merge). Ejecuta las celdas en orden, una por una.

Opcional (comparar v1 vs v2): sube la carpeta local `CerberusAI\finetune\lora-adapter\` (adapter v1)
a Drive como `MyDrive/CerberusAI/runs/v1/lora-adapter/`.

## 0. Parametros

In [ ]:
RUN_ID = "v2-20261106-1"                  # corrida entrenada a exportar/evaluar
BASE_MODEL = "qwen2.5-coder:7b-instruct"  # mismo modelo base que en la evaluacion de v1

REPO = "https://github.com/Coliiin1/CerberusIDE-IA.git"
BRANCH = "main"
LLAMA_REPO = "https://github.com/ggml-org/llama.cpp"
JDK_URL = "https://api.adoptium.net/v3/binary/latest/25/ga/linux/x64/jdk/hotspot/normal/eclipse"

DRIVE = "/content/drive/MyDrive/CerberusAI"
RUN_DIR = DRIVE + "/runs/" + RUN_ID
ADAPTER_V2 = RUN_DIR + "/lora-adapter"
GGUF_V2 = RUN_DIR + "/gguf/cerberusai-q8_0.gguf"
ADAPTER_V1 = DRIVE + "/runs/v1/lora-adapter"
GGUF_V1 = "/content/v1/cerberusai-q8_0.gguf"     # v1 solo vive en el disco de Colab
EVAL_DIR = RUN_DIR + "/eval"

REPO_DIR = "/content/CerberusIDE-IA"
AI_DIR = REPO_DIR + "/CerberusAI"
LLAMA_DIR = "/content/llama.cpp"
JDK_DIR = "/content/jdk"

import subprocess

def run(cmd, cwd=None):
    """Ejecuta un comando mostrando su salida en vivo; se detiene si falla."""
    print("$", " ".join(cmd))
    p = subprocess.Popen(cmd, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    lineas = []
    for linea in p.stdout:
        print(linea, end="")
        lineas.append(linea)
    assert p.wait() == 0, "fallo (exit {}): {}".format(p.returncode, " ".join(cmd))
    return "".join(lineas)

print("RUN_DIR:", RUN_DIR)

## 1. Verificar GPU y RAM

In [ ]:
import os, subprocess
r = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
print(r.stdout or r.stderr)
assert r.returncode == 0, "Sin GPU: Entorno de ejecucion > Cambiar tipo de entorno > A100"
RAM_GB = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1e9
print("RAM: {:.0f} GB".format(RAM_GB))
assert RAM_GB >= 40, "El merge fp16 necesita ~28 GB de RAM: usa A100 (o activa 'RAM alta')."

## 2. Montar Drive y comprobar los adapters

In [ ]:
import os, shutil
from google.colab import drive
drive.mount("/content/drive")

assert os.path.isfile(ADAPTER_V2 + "/adapter_model.safetensors"), "No existe el adapter v2 en " + ADAPTER_V2
HAY_V1 = os.path.isfile(ADAPTER_V1 + "/adapter_model.safetensors")
print("adapter v2: OK")
print("adapter v1:", "OK (se comparara v1 vs v2)" if HAY_V1 else "no subido (solo base vs v2)")

libre = shutil.disk_usage("/content/drive/MyDrive").free / 1e9
print("GGUF v2 ya existe:", os.path.isfile(GGUF_V2))
print("Espacio libre en Drive: {:.1f} GB (el GGUF ocupa ~8 GB)".format(libre))
if not os.path.isfile(GGUF_V2) and libre < 9:
    print("AVISO: puede que no quepa el GGUF en Drive; libera espacio antes del paso 4.")

## 3. Clonar el repo y llama.cpp, instalar dependencias

In [ ]:
import os, subprocess, sys
if not os.path.isdir(REPO_DIR):
    run(["git", "clone", "-b", BRANCH, REPO, REPO_DIR])
if not os.path.isdir(LLAMA_DIR):
    run(["git", "clone", "--depth", "1", LLAMA_REPO, LLAMA_DIR])
COMMIT = subprocess.check_output(["git", "-C", REPO_DIR, "rev-parse", "HEAD"], text=True).strip()
LLAMA_COMMIT = subprocess.check_output(["git", "-C", LLAMA_DIR, "rev-parse", "HEAD"], text=True).strip()
print("CerberusIDE-IA:", COMMIT)
print("llama.cpp:", LLAMA_COMMIT)

run([sys.executable, "-m", "pip", "install", "-q", "-r", AI_DIR + "/colab/requirements-colab.txt"])
run([sys.executable, "-m", "pip", "install", "-q", LLAMA_DIR + "/gguf-py"])
r = subprocess.run([sys.executable, LLAMA_DIR + "/convert_hf_to_gguf.py", "--help"], capture_output=True, text=True)
print((r.stdout + r.stderr)[-800:])
assert r.returncode == 0, "convert_hf_to_gguf.py no arranca: pega este error."

## 4. Merge + GGUF Q8_0

`export_gguf.py` (merge fp16 en CPU, ~5-15 min) y `convert_hf_to_gguf.py --outtype q8_0`
(equivale a f16 + `llama-quantize Q8_0`). El GGUF de v2 se guarda en Drive con su sha256; si ya
existe, se omite (sirve para retomar tras una desconexion).

In [ ]:
import hashlib, json, os, shutil, sys

def exportar(adapter, gguf):
    if os.path.isfile(gguf):
        print("Ya existe, se omite:", gguf)
        return
    merged, tmp = "/content/merged", "/content/tmp-q8_0.gguf"
    shutil.rmtree(merged, ignore_errors=True)
    run([sys.executable, AI_DIR + "/finetune/export_gguf.py", "--adapter", adapter, "--merged-dir", merged])
    run([sys.executable, LLAMA_DIR + "/convert_hf_to_gguf.py", merged, "--outtype", "q8_0", "--outfile", tmp])
    shutil.rmtree(merged)
    os.makedirs(os.path.dirname(gguf), exist_ok=True)
    shutil.move(tmp, gguf)  # se escribe en local y se mueve al final: en Drive nunca queda un GGUF a medias
    print("GGUF:", gguf, round(os.path.getsize(gguf) / 1e9, 2), "GB")

def sha256(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for bloque in iter(lambda: f.read(1 << 24), b""):
            h.update(bloque)
    return h.hexdigest()

exportar(ADAPTER_V2, GGUF_V2)
INFO = os.path.dirname(GGUF_V2) + "/gguf_info.json"
if not os.path.isfile(INFO):
    info = {"run_id": RUN_ID, "archivo": os.path.basename(GGUF_V2), "cuantizacion": "Q8_0",
            "bytes": os.path.getsize(GGUF_V2), "sha256": sha256(GGUF_V2),
            "commit_repo": COMMIT, "commit_llama_cpp": LLAMA_COMMIT}
    with open(INFO, "w", encoding="utf-8") as f:
        json.dump(info, f, indent=2)
with open(INFO, encoding="utf-8") as f:
    print(f.read())
assert os.path.getsize(GGUF_V2) > 7e9, "GGUF demasiado pequeno: algo fallo en la conversion."

if HAY_V1:
    exportar(ADAPTER_V1, GGUF_V1)

## 5. Ollama en Colab

In [ ]:
!apt-get -qq install -y zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
import subprocess, time, urllib.request

OLLAMA = subprocess.Popen(["ollama", "serve"], stdout=open("/content/ollama.log", "w"), stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        urllib.request.urlopen("http://localhost:11434/api/tags", timeout=2)
        break
    except Exception:
        time.sleep(1)
else:
    raise AssertionError("Ollama no arranco: revisa /content/ollama.log")
print("Ollama en marcha.")

def crear(nombre, gguf):
    with open(AI_DIR + "/finetune/Modelfile", encoding="utf-8") as f:
        mf = f.read().replace("FROM ./cerberusai-q8_0.gguf", "FROM " + gguf)
    assert "FROM " + gguf in mf, "El Modelfile ya no tiene 'FROM ./cerberusai-q8_0.gguf'"
    ruta = "/content/Modelfile." + nombre
    with open(ruta, "w", encoding="utf-8") as f:
        f.write(mf)
    run(["ollama", "create", nombre, "-f", ruta])

crear("cerberusai-v2", GGUF_V2)
if HAY_V1:
    crear("cerberusai-v1", GGUF_V1)
run(["ollama", "pull", BASE_MODEL])

lista = run(["ollama", "list"])
for m in ["cerberusai-v2", BASE_MODEL] + (["cerberusai-v1"] if HAY_V1 else []):
    assert m in lista, "Falta el modelo " + m

## 6. Oraculo: JDK 25 + compilar el compilador Cerberus + TestSuite

In [ ]:
import glob, os, subprocess, urllib.request

if not os.path.isdir(JDK_DIR):
    urllib.request.urlretrieve(JDK_URL, "/content/jdk.tar.gz")
    os.makedirs(JDK_DIR)
    run(["tar", "-xzf", "/content/jdk.tar.gz", "-C", JDK_DIR, "--strip-components=1"])
JAVA, JAVAC = JDK_DIR + "/bin/java", JDK_DIR + "/bin/javac"
print(subprocess.run([JAVA, "-version"], capture_output=True, text=True).stderr)

# Solo los paquetes puros (sin JavaFX/Jackson): no hace falta Maven ni module-info.
PAQUETES = ["audit", "AnalizadorLexico", "AnalizadorSintactico", "AnalizadorSemantico", "Util", "Archivos"]
fuentes = [f for p in PAQUETES for f in glob.glob(REPO_DIR + "/src/main/java/" + p + "/**/*.java", recursive=True)]
run([JAVAC, "-encoding", "UTF-8", "-d", REPO_DIR + "/target/classes", *fuentes])

r = subprocess.run([JAVA, "-cp", "target/classes", "audit.TestSuite"], cwd=REPO_DIR, capture_output=True, text=True)
print(r.stdout[-400:])
assert r.returncode == 0, "TestSuite fallo: el oraculo no es fiable."

## 7. Evaluacion

Tareas `generation` de `CerberusAI/finetune/eval.jsonl` (las mismas 104 que se apartaron en el
entrenamiento; se comprueba con el sha256 de `run_info.json`). Resultados en `runs/<RUN_ID>/eval/`.

In [ ]:
import hashlib, json, os, sys

with open(RUN_DIR + "/run_info.json", encoding="utf-8") as f:
    esperado = json.load(f)["sha256"]["eval.jsonl"]
with open(AI_DIR + "/finetune/eval.jsonl", "rb") as f:
    assert hashlib.sha256(f.read()).hexdigest() == esperado, "eval.jsonl del repo != el usado al entrenar"
print("eval.jsonl identico al de la corrida.")

os.makedirs(EVAL_DIR, exist_ok=True)

def evaluar(base, ft, nombre):
    salida = run([sys.executable, AI_DIR + "/eval/run_eval.py", "--base-model", base, "--ft-model", ft,
                  "--java", JAVA])
    with open(EVAL_DIR + "/" + nombre, "w", encoding="utf-8") as f:
        f.write(salida)
    return salida

RESULTADOS = {"resultados_base_vs_v2.txt": evaluar(BASE_MODEL, "cerberusai-v2", "resultados_base_vs_v2.txt")}
if HAY_V1:
    RESULTADOS["resultados_v1_vs_v2.txt"] = evaluar("cerberusai-v1", "cerberusai-v2", "resultados_v1_vs_v2.txt")

## 8. Resumen

In [ ]:
for nombre, salida in RESULTADOS.items():
    print("=" * 20, nombre)
    print(salida[salida.find("===== COMPARACION"):])
print("Guardado en:", EVAL_DIR)

## 9. Si v2 gana: servirlo en el PC

Es el unico paso en tu equipo (el IDE usa Ollama en local):

1. Descarga `runs/<RUN_ID>/gguf/cerberusai-q8_0.gguf` desde la web de Google Drive
   (~8 GB) a `CerberusAI\finetune\` (reemplaza al GGUF v1; v1 sigue registrado en Ollama).
2. Comprueba el sha256 contra `gguf_info.json`: `certutil -hashfile cerberusai-q8_0.gguf SHA256`.
3. `ollama cp cerberusai cerberusai-v1` (respaldo de v1).
4. `cd CerberusAI\finetune` y `ollama create cerberusai -f Modelfile`.

La API y el chat del IDE siguen usando `cerberusai`, sin cambios de codigo.